In [1]:
!nvidia-smi

Tue Apr 22 03:29:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:23:00.0 Off |                    0 |
|  0%   37C    P0             57W /  300W |       4MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers pillow

In [3]:
import os
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image

# Reduce memory fragmentation in PyTorch allocations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/dserrano/miniconda3/envs/resshift/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceTB/SmolVLM-Instruct",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    # Remove or set to "eager" or "sdpa"
    _attn_implementation="eager"
).to(DEVICE)

Using device: cuda


In [5]:
from glob import glob
image_paths = glob("images/apt1/*.jpg")

images = [load_image(path) for path in image_paths]

In [18]:
messages = [
    {
        "role": "user",
        "content": (
            [{"type": "image"} for _ in images] +
            [{"type": "text", "text": "Describe this apartment for a real state website."}]
        )
    }
]

In [19]:
def clear_cache():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

In [20]:
clear_cache()

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=images, return_tensors="pt").to(DEVICE)

# Generate the description
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=500)
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

print("Apartment Description:\n", generated_texts[0])

Apartment Description:
 User:<image>Describe this apartment for a real state website.
Assistant: The apartment is a small, white, and gray space with a gray tile floor. The room is divided into two areas: a living room and a kitchen. The living room has a white desk with a laptop, a chair, and a backpack on it. There is a black backpack on the floor next to the desk. The kitchen has white cabinets and a white refrigerator. There is a mirror on the wall in the kitchen. There is a black backpack hanging on the wall next to the refrigerator. There is a black t-shirt hanging on the wall next to the door. There is a white door next to the t-shirt. There is a gray couch in the living room. There is a black backpack on the floor next to the couch. There is a white door next to the couch. There is a gray rug in front of the couch. There is a white light fixture on the ceiling. There is a white light fixture on the wall. There is a white light fixture on the floor.
